# Pseudo-Relevance Feedback (PRF)
---
[[paper]](https://dl.acm.org/doi/10.1145/358968.358994) (Classic RM3 paper, 2001) <br>
[[original_concept]](https://dl.acm.org/doi/10.1145/1221.1223) (Rocchio Algorithm, 1971) <br>
PRF = Pseudo-Relevance Feedback

__PRF__ — это алгоритмический метод автоматического уточнения поискового запроса, который основывается на предположении, что несколько наиболее релевантных документов (Top-K) из результатов первичного поиска содержат важные термины, описывающие интент пользователя. Метод позволяет решить проблему Vocabulary Mismatch без участия человека в процессе разметки.

__Постановка задачи__<br>
Пользователь подает короткий и неполный запрос $Q$. Необходимо вернуть список документов $D$, отранжированных по релевантности, минимизируя пропуски релевантных документов, которые не содержат точных слов из запроса.

__Мотивация__<br>
Основная проблема поиска — Vocabulary Mismatch (лексический разрыв). Пользователь может искать "лечение мигрени", в то время как в наиболее качественных медицинских статьях используются термины "терапия" и "гемикрания". Обычный Sparse Retrieval (BM25, TF-IDF) не найдет эти документы, так как в них нет пересечения по токенам. PRF пытается "догадаться", какие еще слова могли бы дополнить запрос, глядя на результаты первого этапа поиска.

__Существующие подходы__<br>
До появления и популяризации PRF (и в частности его эффективных реализаций вроде RM3 в 2001 году) существовали:
- Relevance Feedback (1971): классический метод Роккио, требующий от пользователя явно отметить, какие документы релевантны, а какие нет. Это работает точно, но пользователи редко готовы давать фидбек.
- Thesaurus-based Expansion: расширение запроса с помощью синонимов из внешних словарей (WordNet). Проблема в том, что словари статичны и не учитывают контекст конкретной коллекции документов.
- Global Analysis: статистический анализ всей коллекции документов для поиска ассоциаций между словами. Это вычислительно дорого и плохо адаптируется под специфические темы запросов.

__Идея__<br>
Вместо того чтобы просить пользователя разметить результаты, мы "слепо" (Blind Feedback) предполагаем, что первые $K$ документов в выдаче являются релевантными. Мы извлекаем из этих документов наиболее характерные термины и добавляем их к исходному запросу, смещая вектор запроса в сторону "центра масс" релевантной информации.

__Архитектура__<br>
PRF не является отдельной моделью с весами, это процедурный фреймворк, встраиваемый в Retrieval Pipeline. Он состоит из следующих компонентов:
1.  Ranker (Initial Retrieval): базовый движок поиска (например, BM25).
2.  Feedback Pool: буфер, содержащий Top-K документов первого этапа.
3.  Term Selector: модуль анализа важности слов в документах фидбека (использует TF-IDF, KL-divergence или Log-odds ratio).
4.  Query Reformulator: механизм объединения исходного запроса и новых терминов.

__Алгоритм работы__<br>
Метод работает исключительно на этапе инференса (или выполнения запроса), так как он адаптируется под каждый конкретный Query.

1.  Initial Retrieval: Выполняется поиск по исходному запросу $Q$. Получаем список $R = \{d_1, d_2, \ldots, d_n\}$.
2.  Sampling: Отбираются первые $K$ документов (обычно $K \in [5, 50]$).
3.  Feature Extraction: Для всех слов в этих документах рассчитывается их "информативность" относительно данного набора. В модели RM3 (2001) это делается через оценку вероятности появления слова в языковой модели релевантности $P(w|R)$.
4.  Expansion: Выбираются Top-M слов с наибольшим весом. Эти слова добавляются в запрос.
5.  Re-weighting: Итоговый запрос формируется как линейная комбинация: $Q_{new} = \alpha \cdot Q_{old} + (1 - \alpha) \cdot Q_{expanded}$. Параметр $\alpha$ контролирует, насколько сильно мы доверяем исходным словам пользователя.
6.  Final Retrieval: Выполняется повторный поиск с использованием $Q_{new}$ по всей коллекции.

__Алгоритм обучения__<br>
В классическом PRF (Rocchio, RM3) обучения в смысле градиентного спуска нет. Однако существуют гиперпараметры, которые оптимизируются на валидационной выборке:
- $K$ (Feedback Documents): количество документов для анализа.
- $M$ (Expansion Terms): количество добавляемых слов.
- $\alpha$ (Original Weight): вес исходного запроса.
В современных версиях (Neural PRF, 2017+) в качестве Term Selector используется нейронная сеть, которая обучается предсказывать веса терминов, минимизируя Cross-Entropy между распределением терминов в расширенном запросе и распределением в идеально релевантном документе.

__Результаты__<br>
Метод PRF является одним из самых стабильных способов поднять качество поиска:
- Mean Average Precision (MAP) увеличивается на 10-20% по сравнению с базовым BM25 на датасетах TREC.
- Recall@1000 растет за счет нахождения документов, не содержащих исходных слов запроса (решение Vocabulary Mismatch).
- Основной риск (Query Drift): если первые $K$ документов оказались нерелевантными ("мусор на входе"), PRF только ухудшит результат, уводя поиск еще дальше в сторону нерелевантной темы. Это привело к появлению методов фильтрации фидбека.

## 📝 Критический анализ

```markdown
# Pseudo-Relevance Feedback (PRF)
---
[[paper]](https://dl.acm.org/doi/10.1145/358968.358994) (Classic RM3 paper, 2001)  
[[original_concept]](https://dl.acm.org/doi/10.1145/1221.1223) (Rocchio Algorithm, 1971)  

**PRF** — алгоритм автоматического уточнения поискового запроса, предполагающий, что несколько наиболее релевантных документов (Top-K) содержат важные термины, описывающие интент пользователя. Метод решает проблему Vocabulary Mismatch без участия человека.

__Постановка задачи__  
Пользователь подает короткий запрос $Q$. Необходимо вернуть список документов $D$, отранжированных по релевантности, минимизируя пропуски документов, не содержащих точных слов из запроса.

__Мотивация__  
Основная проблема — Vocabulary Mismatch. Например, пользователь ищет "лечение мигрени", а в статьях используются "терапия" и "гемикрания". PRF пытается дополнить запрос, анализируя результаты первого этапа поиска.

__Существующие подходы__  
До PRF существовали:
- Relevance Feedback (1971): требовал явной разметки релевантности от пользователя.
- Thesaurus-based Expansion: использовал статичные словари, не учитывающие контекст.
- Global Analysis: статистический анализ всей коллекции, вычислительно дорогой.

__Идея__  
Вместо разметки пользователем, предполагаем, что первые $K$ документов релевантны. Извлекаем из них термины и добавляем к запросу, смещая его вектор к "центру масс" релевантной информации.

__Архитектура__  
PRF — процедурный фреймворк в Retrieval Pipeline:
1. Ranker: базовый движок поиска (например, BM25).
2. Feedback Pool: буфер с Top-K документами.
3. Term Selector: анализ важности слов (TF-IDF, KL-divergence).
4. Query Reformulator: объединяет исходный запрос и новые термины.

<img src="img/img.png" width=500>

__Алгоритм работы__  
Метод работает на этапе инференса, адаптируясь под каждый Query.

1. Initial Retrieval: поиск по $Q$, получаем список $R$.
2. Sampling: отбираем первые $K$ документов.
3. Feature Extraction: рассчитываем "информативность" слов в документах.
4. Expansion: выбираем Top-M слов, добавляем в запрос.
5. Re-weighting: формируем $Q_{new}$ как линейную комбинацию: $Q_{new} = \alpha \cdot Q_{old} + (1 - \alpha) \cdot Q_{expanded}$.
6. Final Retrieval: повторный поиск с $Q_{new}$.

__Алгоритм обучения__  
В классическом PRF обучения нет, но оптимизируются гиперпараметры: $K$, $M$, $\alpha$. В современных версиях (Neural PRF, 2017+) используется нейронная сеть для предсказания весов терминов.

__Результаты__  
PRF стабильно улучшает качество поиска:
- Mean Average Precision (MAP) увеличивается на 10-20% по сравнению с BM25 на TREC.
- Recall@1000 растет за счет нахождения документов, не содержащих исходных слов запроса.
- Риск Query Drift: нерелевантные $K$ документы ухудшают результат, что привело к методам фильтрации фидбека.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Пример коллекции документов
documents = [
    "Migraine treatment includes medication and lifestyle changes.",
    "Therapy for hemicrania involves various approaches.",
    "Migraine is a common type of headache.",
    "Hemicrania is another term for migraine.",
    "Effective migraine therapy can improve quality of life."
]

# Исходный запрос пользователя
query = "migraine treatment"

# Шаг 1: Initial Retrieval с использованием TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])

# Вычисляем косинусное сходство между запросом и документами
cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

# Получаем индексы Top-K документов
K = 3
top_k_indices = cosine_similarities.argsort()[-K:][::-1]

# Шаг 2: Sampling - отбираем Top-K документов
feedback_pool = [documents[i] for i in top_k_indices]

# Шаг 3: Feature Extraction - вычисляем важность терминов
# Объединяем документы из feedback_pool в один текст
feedback_text = " ".join(feedback_pool)
feedback_vector = vectorizer.transform([feedback_text])

# Вычисляем вероятности появления слов в языковой модели релевантности
term_importance = feedback_vector.toarray().flatten()

# Шаг 4: Expansion - выбираем Top-M терминов для расширения запроса
M = 2
top_m_indices = term_importance.argsort()[-M:][::-1]
expansion_terms = [vectorizer.get_feature_names_out()[i] for i in top_m_indices]

# Шаг 5: Re-weighting - формируем новый запрос
alpha = 0.7
expanded_query_terms = query.split() + expansion_terms
expanded_query = " ".join(expanded_query_terms)

# Шаг 6: Final Retrieval - повторный поиск с расширенным запросом
expanded_query_vector = vectorizer.transform([expanded_query])
final_cosine_similarities = cosine_similarity(expanded_query_vector, tfidf_matrix).flatten()

# Сортируем документы по релевантности
final_ranking_indices = final_cosine_similarities.argsort()[::-1]

# Выводим результаты
print("Исходный запрос:", query)
print("Расширенные термины:", expansion_terms)
print("Новый запрос:", expanded_query)
print("\nРанжированные документы:")
for idx in final_ranking_indices:
    print(f"Документ: {documents[idx]} (Сходство: {final_cosine_similarities[idx]:.4f})")
```

### Объяснение ключевых моментов:

1. **Initial Retrieval**: Используем TF-IDF для первичного поиска, чтобы получить Top-K документов, которые наиболее релевантны исходному запросу.

2. **Sampling**: Отбираем первые K документов из результатов первичного поиска для дальнейшего анализа.

3. **Feature Extraction**: Вычисляем важность терминов в отобранных документах, используя TF-IDF. Это позволяет определить, какие термины могут быть полезны для расширения запроса.

4. **Expansion**: Выбираем Top-M терминов с наибольшей важностью и добавляем их к исходному запросу.

5. **Re-weighting**: Формируем новый запрос как комбинацию исходного и расширенного запросов, контролируя влияние каждого с помощью параметра α.

6. **Final Retrieval**: Выполняем повторный поиск с расширенным запросом, чтобы улучшить результаты и минимизировать проблему Vocabulary Mismatch.

Этот пример иллюстрирует, как PRF может быть реализован с использованием простых инструментов, таких как TF-IDF, для улучшения качества поиска.